# Streamlit UI Smoke Test

Checks the UI module as source code without importing it directly. Importing `src/ui/app.py` would execute Streamlit page code, so this notebook validates launch command, dependencies, and expected UI hooks safely.

In [ ]:
from pathlib import Path
import importlib.metadata as md
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
ui_file = project_root / 'src' / 'ui' / 'app.py'
print('project_root:', project_root)
print('ui_file:', ui_file)
print('exists:', ui_file.exists())
assert ui_file.exists()

In [ ]:
import streamlit as st

print('streamlit version:', md.version('streamlit'))
print('launch command:')
print('  uv run streamlit run src/ui/app.py')
print('docker URL: http://localhost:8501')
assert st is not None

In [ ]:
source = ui_file.read_text(encoding='utf-8', errors='replace')

required_snippets = [
    'st.set_page_config',
    'st.session_state.thread_id',
    'def run_query',
    'copilot_graph.invoke',
    'st.chat_input',
    'GraphResponse',
]

for snippet in required_snippets:
    present = snippet in source
    print(f'{snippet:30s}', present)
    assert present

In [ ]:
# Extract a compact UI feature inventory from source.
features = {
    'chat history': 'chat_history' in source,
    'thread memory': 'thread_id' in source,
    'mock mode toggle': 'Mock Mode' in source,
    'time range selector': 'Time Range' in source,
    'example queries': 'Try these' in source,
    'raw metrics expander': 'Raw metrics' in source,
    'sources expander': 'Sources' in source,
    'agents used expander': 'Agents used' in source,
}

for name, enabled in features.items():
    print(f'{name:22s}', enabled)
    assert enabled